# TabPFN-3 fair test on a free 16 GB GPU (Google Colab)

**Before running (once):**
1. **Google Drive:** create a folder `stock_tabpfn` in *My Drive* and upload these 5 files into it:
   `train_matrix.csv.gz`, `validation_matrix.csv.gz`, `test_matrix.csv.gz` (from `data\split`),
   `sectors.csv` (from `data\global`) and `run_tabpfn3.py`.
2. **GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
3. **Key:** click the **key icon** (Secrets) in the left bar → **Add new secret** →
   Name `TABPFN_TOKEN`, Value = your Prior Labs API key → switch **Notebook access** on.

**Then:** menu **Runtime → Run all**. Keep this browser tab open until it finishes
(free Colab stops if the tab is closed or idle too long). Results are saved to your Drive folder.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE = '/content/drive/MyDrive/stock_tabpfn'
need = ['train_matrix.csv.gz', 'validation_matrix.csv.gz', 'test_matrix.csv.gz', 'run_tabpfn3.py']
missing = [f for f in need if not os.path.exists(os.path.join(DRIVE, f))]
assert not missing, f'Missing in {DRIVE}: {missing}'

# Copy to the fast local disk
DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)
for f in os.listdir(DRIVE):
    if f.endswith(('.csv.gz', '.csv', '.py')):
        shutil.copy(os.path.join(DRIVE, f), DATA)
SECTORS = os.path.join(DATA, 'sectors.csv')
print(os.listdir(DATA))
print('Sectors:', 'found' if os.path.exists(SECTORS) else 'not found (sector features skipped)')

In [ ]:
!pip install -q --upgrade tabpfn lightgbm

In [ ]:
import os
from google.colab import userdata
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN')
os.environ['TABPFN_NO_BROWSER'] = '1'
print('Token loaded:', bool(os.environ['TABPFN_TOKEN']))

## Run 1: large context (fixed fit)
200,000 context rows (5x your laptop). If the GPU runs out of memory, the script halves the rows automatically.
`--eval-every 5` predicts every 5th trading day: about 5x faster, same stock-picking test.

In [ ]:
!cd /content && python data/run_tabpfn3.py --data-dir "$DATA" --sectors "$SECTORS" \
    --refit-on-train-val --context-rows 200000 --n-estimators 4 \
    --prediction-batch-size 5000 --eval-every 5 \
    --output /content/results_200k.json --predictions /content/predictions_200k.csv.gz
!cp /content/results_200k.json /content/predictions_200k.csv.gz "$DRIVE"/ 2>/dev/null; echo "Saved to Drive: $DRIVE" 

## Run 2 (optional, if the session is still alive): 200k rows sampled from 3 years

In [ ]:
!cd /content && python data/run_tabpfn3.py --data-dir "$DATA" --sectors "$SECTORS" \
    --refit-on-train-val --context-rows 200000 --context-years 3 --n-estimators 4 \
    --prediction-batch-size 5000 --eval-every 5 \
    --output /content/results_200k_3y.json --predictions /content/predictions_200k_3y.csv.gz
!cp /content/results_200k_3y.json /content/predictions_200k_3y.csv.gz "$DRIVE"/ 2>/dev/null; echo "Saved to Drive: $DRIVE" 

In [ ]:
import json, os
for f in ['results_200k.json', 'results_200k_3y.json']:
    p = os.path.join('/content', f)
    if not os.path.exists(p):
        continue
    r = json.load(open(p))
    print(f"\n{f}: context rows used {r['context_rows_used']:,}, features {r['n_features']}")
    for split in ('validation', 'test'):
        for model, m in r[split].items():
            print(f"  {split:10s} {model:20s} AUC {m['roc_auc']:.2f}  within-day {m['within_day_auc']:.2f}  "
                  f"top-bottom {m['top_minus_bottom_pct']:+.3f}%  t {m['t_stat']:+.2f}")